# Blind-test evaluation and error analysis
Formal metrics use only verified legitimate and generated phishing rows. Weak-feed captures are shown only as a stress distribution.

In [ ]:
%pip install -r ../requirements-notebook.txt

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir(): ROOT = ROOT.parent
rf = json.loads((ROOT/'artifacts/v3/detector_v3_metrics.json').read_text(encoding='utf-8'))
ens = json.loads((ROOT/'artifacts/v3/url_tcn_metrics.json').read_text(encoding='utf-8'))
table = pd.DataFrame({
 'RF 0.5': rf['test_at_0_5'],
 'RF policy': rf['test_at_policy_threshold'],
 'TCN 0.5': ens['tcn_test_at_0_5'],
 'Ensemble policy': ens['ensemble_test_at_policy_threshold'],
}).T
display(table[['precision','recall','f1','false_positive_rate','roc_auc','pr_auc']])

In [ ]:
pred = pd.read_csv(ROOT/'artifacts/v3/url_tcn_test_predictions.csv')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for label, group in pred.groupby('label'):
    group['rf_probability'].plot.hist(ax=axes[0], bins=30, alpha=.55, label=str(label), density=True)
    group['tcn_probability'].plot.hist(ax=axes[1], bins=30, alpha=.55, label=str(label), density=True)
    group['combined_probability'].plot.hist(ax=axes[2], bins=30, alpha=.55, label=str(label), density=True)
for ax, title in zip(axes, ['RF', 'TCN', 'Combined']):
    ax.set_title(title); ax.set_xlabel('phishing probability'); ax.legend(title='label')
plt.tight_layout()

In [ ]:
threshold = ens['ensemble_policy']['phishing_threshold']
false_positives = pred[(pred.label == 0) & (pred.combined_probability >= threshold)].sort_values('combined_probability', ascending=False)
false_negatives = pred[(pred.label == 1) & (pred.combined_probability < threshold)].sort_values('combined_probability')
print('False positives:', len(false_positives), 'False negatives:', len(false_negatives))
display(false_positives.head(20))
display(false_negatives.head(20))

In [ ]:
weak = pd.read_csv(ROOT/'artifacts/v3/detector_v3_weak_stress_predictions.csv')
display(weak['probability'].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]).to_frame('weak-feed score'))
print('Do not calculate precision/recall here: these are broad blocklist labels, not adjudicated phishing truth.')